# Junaid's Cross-Modal Reproduction Notebook (CUDA)

This notebook is for **re-running the SHD-20 + N-Caltech101 experiment sweep on CUDA** and validating the outputs.

It follows a code-first, multi-cell workflow so each stage can be executed and monitored separately:

1. Runtime / dependency setup
2. Repo + dataset path setup
3. Model factory + pipeline sanity checks
4. Sweep execution (`run_experiments.py`)
5. Cross-modal validation (temporal engram curves, energy ratio, accuracy)


In [ ]:
"""Cell 1: GPU Environment Check"""
from __future__ import annotations

import platform
import subprocess
import sys
from pathlib import Path

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Working directory:', Path.cwd())

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('CUDA device count:', torch.cuda.device_count())
        print('CUDA device[0]:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('PyTorch import/check failed:', exc)

try:
    proc = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=False)
    print('nvidia-smi exit code:', proc.returncode)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
except FileNotFoundError:
    print('nvidia-smi not found in this runtime.')


In [ ]:
"""Cell 2: Install Required Dependencies (Conditional)"""
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path

# Leave torch alone by default so the CUDA runtime build is not overwritten.
INSTALL_TORCH = False
AUTO_INSTALL_MISSING = True
USE_REQUIREMENTS_FILE = True
FULL_REQUIREMENTS_SYNC = False  # Safer default: only install missing packages unless explicitly enabled.

requirements_path = None
for candidate in [Path.cwd()/"requirements.txt", Path.cwd().parent/"requirements.txt"]:
    if candidate.exists():
        requirements_path = candidate
        break

module_to_pip = {
    'numpy': 'numpy<2.0',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'sklearn': 'scikit-learn',
    'tqdm': 'tqdm',
    'h5py': 'h5py',
    'scipy': 'scipy',
    'snntorch': 'snntorch',
    'tonic': 'tonic',
    'wandb': 'wandb',
    'IPython': 'ipython',
    'nbformat': 'nbformat',
    'nbclient': 'nbclient',
}
if INSTALL_TORCH:
    module_to_pip['torch'] = 'torch'

missing = [pkg for mod, pkg in module_to_pip.items() if importlib.util.find_spec(mod) is None]
print('Missing packages:', missing if missing else 'None')
print('requirements.txt found:', requirements_path)
print('FULL_REQUIREMENTS_SYNC:', FULL_REQUIREMENTS_SYNC)

if AUTO_INSTALL_MISSING:
    if missing:
        # Default path: install only what is missing.
        cmd = [sys.executable, '-m', 'pip', 'install'] + missing
        print('Running:', ' '.join(cmd))
        subprocess.check_call(cmd)
    elif USE_REQUIREMENTS_FILE and FULL_REQUIREMENTS_SYNC and requirements_path is not None:
        # Optional path: reconcile/sync the full notebook environment from requirements.txt.
        cmd = [sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)]
        print('Running full requirements sync:', ' '.join(cmd))
        subprocess.check_call(cmd)
    else:
        print('Nothing to install.')
else:
    print('AUTO_INSTALL_MISSING=False. Install dependencies manually if needed.')


In [ ]:
"""Cell 3: Workspace and Path Setup"""
from __future__ import annotations

from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "run_experiments.py").exists() and (p / "training").exists() and (p / "Models").exists():
            return p
    raise FileNotFoundError('Could not locate repo root from current directory.')

REPO_ROOT = find_repo_root(Path.cwd())
NOTEBOOK_DIR = REPO_ROOT / "notebook"
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
DATA_DIR = REPO_ROOT / "data"
CHECKPOINTS_DIR = REPO_ROOT / "checkpoints"
for p in [RESULTS_DIR, FIGURES_DIR, DATA_DIR, CHECKPOINTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT =', REPO_ROOT)
print('RESULTS_DIR =', RESULTS_DIR)
print('FIGURES_DIR =', FIGURES_DIR)
print('DATA_DIR =', DATA_DIR)
print('CHECKPOINTS_DIR =', CHECKPOINTS_DIR)


In [ ]:
"""Cell 4: Core Imports and Visualization Defaults"""
from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Models.registry import build_model
from training.pipeline import (
    create_ann_baseline,
    make_energy_methodology_callback,
    make_temporal_silhouette_callback,
    run_seed_iterator,
)
from visualization import plot_engram_evolution

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'legend.frameon': False,
})

print('Imports loaded successfully.')
print('plot_engram_evolution available:', callable(plot_engram_evolution))


In [ ]:
"""Cell 5: Dataset and Sweep Configuration"""
from __future__ import annotations

# Execution control
RUN_EXPERIMENTS = False
DEVICE = "cuda"
DATASETS = ["ncaltech101", "shd"]

# Profiles
FAST_PILOT = True
if FAST_PILOT:
    NUM_RUNS = 1
    NUM_EPOCHS = 6
    BATCH_SIZE = 16
    NUM_WORKERS = 4
else:
    NUM_RUNS = 5
    NUM_EPOCHS = 30
    BATCH_SIZE = 16
    NUM_WORKERS = 4

# Reproducibility and optimizer settings
BASE_SEED = 42
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4

# Optional output naming
TIMESTAMPED_OUTPUT = False
PREFER_LATEST_TIMESTAMPED_RESULTS = True
REQUIRE_FULL_CROSS_MODAL_RESULTS = True

# Energy ratio target for quick validation checks
TARGET_EFFICIENCY_RATIO = 603.0

config_snapshot = {
    'RUN_EXPERIMENTS': RUN_EXPERIMENTS,
    'DEVICE': DEVICE,
    'DATASETS': DATASETS,
    'FAST_PILOT': FAST_PILOT,
    'NUM_RUNS': NUM_RUNS,
    'NUM_EPOCHS': NUM_EPOCHS,
    'BATCH_SIZE': BATCH_SIZE,
    'NUM_WORKERS': NUM_WORKERS,
    'BASE_SEED': BASE_SEED,
    'LEARNING_RATE': LEARNING_RATE,
    'WEIGHT_DECAY': WEIGHT_DECAY,
    'TARGET_EFFICIENCY_RATIO': TARGET_EFFICIENCY_RATIO,
    'PREFER_LATEST_TIMESTAMPED_RESULTS': PREFER_LATEST_TIMESTAMPED_RESULTS,
    'REQUIRE_FULL_CROSS_MODAL_RESULTS': REQUIRE_FULL_CROSS_MODAL_RESULTS,
}
display(pd.DataFrame([config_snapshot]))


In [ ]:
"""Cell 6: Dataset Cache and Loader Prerequisite Check"""
from __future__ import annotations

import importlib
from pathlib import Path

print("DATA_DIR:", DATA_DIR)
if DATA_DIR.exists():
    top_items = sorted(DATA_DIR.iterdir())[:30]
    print("Top-level data entries (up to 30):")
    for p in top_items:
        kind = "dir" if p.is_dir() else "file"
        print(f"- [{kind}] {p.name}")
else:
    print("DATA_DIR does not exist yet (it will be created/downloaded by tonic loaders).")

try:
    tonic = importlib.import_module("tonic")
    print("tonic version:", getattr(tonic, "__version__", "unknown"))
except Exception as exc:
    print("tonic import failed:", exc)


In [ ]:
"""Cell 7: Model Factory and Energy/Temporal Pipeline Sanity Check"""
from __future__ import annotations

def count_params(model):
    return sum(p.numel() for p in model.parameters())

models = {}
models["shd_baseline"] = build_model(
    variant="baseline",
    input_type="shd",
    input_size=700,
    num_classes=20,
)
models["ncaltech_baseline"] = build_model(
    variant="baseline",
    input_type="dvs_gesture",
    input_channels=2,
    spatial_size=(34, 34),
    num_classes=101,
)

rows = []
for name, model in models.items():
    ann_shadow = create_ann_baseline(model)
    rows.append({
        'model': name,
        'snn_params': count_params(model),
        'ann_shadow_params': count_params(ann_shadow),
        'param_match': count_params(model) == count_params(ann_shadow),
    })
display(pd.DataFrame(rows))

print('Temporal callback factory available:', callable(make_temporal_silhouette_callback))
print('Energy callback factory available:', callable(make_energy_methodology_callback))
print('Seed iterator available:', callable(run_seed_iterator))


In [ ]:
"""Cell 8: Build Sweep Command"""
from __future__ import annotations

from pathlib import Path
import shlex
import sys

RUNNER_PATH = REPO_ROOT / "run_experiments.py"
if not RUNNER_PATH.exists():
    raise FileNotFoundError(f"Missing runner: {RUNNER_PATH}")

sweep_cmd = [
    sys.executable, str(RUNNER_PATH),
    "--device", DEVICE,
    "--datasets", *DATASETS,
    "--num-workers", str(NUM_WORKERS),
    "--num-runs", str(NUM_RUNS),
    "--num-epochs", str(NUM_EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--base-seed", str(BASE_SEED),
    "--learning-rate", str(LEARNING_RATE),
    "--weight-decay", str(WEIGHT_DECAY),
]
if TIMESTAMPED_OUTPUT:
    sweep_cmd.append("--timestamped-output")

print('Command preview:')
print(' '.join(shlex.quote(x) for x in sweep_cmd))


In [ ]:
"""Cell 9: Execute Sweep (Streams run_experiments.py Output)"""
from __future__ import annotations

import subprocess

if not RUN_EXPERIMENTS:
    print('RUN_EXPERIMENTS=False. Set it to True in Cell 5, rerun Cells 8-9 to launch the sweep.')
else:
    proc = subprocess.Popen(
        sweep_cmd,
        cwd=str(REPO_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    try:
        for line in proc.stdout:
            print(line, end="")
    finally:
        ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"run_experiments.py exited with code {ret}")
    print('Sweep completed successfully.')


In [ ]:
"""Cell 10: Inspect Latest Logs and JSON Artifacts"""
from __future__ import annotations

log_path = REPO_ROOT / "experiment_log.txt"
if log_path.exists():
    lines = log_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    print('Last 60 log lines:')
    for line in lines[-60:]:
        print(line)
else:
    print('experiment_log.txt not found yet.')

print('JSON artifacts in results/:')
for p in sorted((REPO_ROOT / "results").glob("*.json")):
    print("-", p.name)


## Validation and Cross-Modal Analysis

The cells below validate that both datasets ran and perform cross-modal comparisons from the generated JSON artifacts.


In [ ]:
"""Cell 11: Load Result JSON Files"""
from __future__ import annotations

from pathlib import Path

RESULTS_DIR = Path(RESULTS_DIR)


def _latest_matching(path_dir: Path, pattern: str):
    candidates = [p for p in path_dir.glob(pattern) if p.is_file()]
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)


def _resolve_results_pair(results_dir: Path):
    default_final = results_dir / "final_specs.json"
    default_sweep = results_dir / "research_sweep_results.json"

    # Prefer the standard non-timestamped outputs if present.
    if default_final.exists() and default_sweep.exists() and not PREFER_LATEST_TIMESTAMPED_RESULTS:
        return default_final, default_sweep

    latest_sweep = _latest_matching(results_dir, "*research_sweep_results.json")
    latest_final = _latest_matching(results_dir, "*final_specs.json")

    # If both standard files exist and they are as recent as anything else, use them.
    if default_final.exists() and default_sweep.exists() and latest_sweep is not None and latest_final is not None:
        if not PREFER_LATEST_TIMESTAMPED_RESULTS:
            return default_final, default_sweep
        # Prefer timestamped only when explicitly requested; otherwise fall back to standard names.
        if latest_sweep.name == default_sweep.name and latest_final.name == default_final.name:
            return default_final, default_sweep

    if latest_sweep is None and latest_final is None:
        raise FileNotFoundError(f"No result JSON files found in {results_dir}")

    # Try to pair by shared prefix (timestamped runs write both files with same prefix).
    if latest_sweep is not None:
        sweep_suffix = "research_sweep_results.json"
        prefix = latest_sweep.name[:-len(sweep_suffix)]
        paired_final = results_dir / f"{prefix}final_specs.json"
        if paired_final.exists():
            return paired_final, latest_sweep

    # Fallback: choose latest of each type independently.
    if latest_final is None:
        raise FileNotFoundError(f"Could not locate any *final_specs.json in {results_dir}")
    if latest_sweep is None:
        raise FileNotFoundError(f"Could not locate any *research_sweep_results.json in {results_dir}")
    return latest_final, latest_sweep


FINAL_SPECS_PATH, SWEEP_RESULTS_PATH = _resolve_results_pair(RESULTS_DIR)
print('Resolved FINAL_SPECS_PATH =', FINAL_SPECS_PATH)
print('Resolved SWEEP_RESULTS_PATH =', SWEEP_RESULTS_PATH)

with open(FINAL_SPECS_PATH, "r", encoding="utf-8") as f:
    final_specs = json.load(f)
with open(SWEEP_RESULTS_PATH, "r", encoding="utf-8") as f:
    sweep_results = json.load(f)

print('Loaded final_specs entries:', list(final_specs.keys()))
print('Sweep meta keys:', list((sweep_results.get('meta') or {}).keys()))
print('Experiment entries:', len((sweep_results.get('experiments') or {})))


In [ ]:
"""Cell 12: Normalize Sweep Results into a Validation Table"""
from __future__ import annotations

def _stat_mean(value, default=np.nan):
    if isinstance(value, dict):
        return float(value.get("mean", default))
    if value is None:
        return float(default)
    return float(value)

def _stat_std(value, default=0.0):
    if isinstance(value, dict):
        return float(value.get("std_dev", value.get("std", default)))
    return float(default)

def iter_experiments(sweep_payload):
    exps = sweep_payload.get("experiments", {})
    if isinstance(exps, dict):
        for k, v in exps.items():
            yield k, v
    elif isinstance(exps, list):
        for idx, v in enumerate(exps):
            key = v.get("tag") or v.get("experiment_key") or f"exp_{idx}"
            yield key, v

rows = []
experiment_map = {}
for exp_key, exp in iter_experiments(sweep_results):
    if not isinstance(exp, dict):
        continue
    experiment_map[exp_key] = exp
    spec = exp.get("final_specs", {})
    temporal = spec.get("temporal_silhouette", {})
    comp = spec.get("compute_comparison", {})
    rows.append({
        'experiment_key': exp_key,
        'config_name': exp.get('config_name'),
        'dataset_name': exp.get('dataset_name'),
        'modality': exp.get('modality'),
        'num_runs': exp.get('num_runs', spec.get('num_runs')),
        'accuracy_mean': _stat_mean(spec.get('accuracy')),
        'accuracy_std': _stat_std(spec.get('accuracy')),
        'energy_ratio_mean': _stat_mean(comp.get('energy_improvement_ratio')),
        'energy_ratio_std': _stat_std(comp.get('energy_improvement_ratio')),
        'snn_synops_mean': _stat_mean(((comp.get('mean_synops_vs_mean_flops_per_sample') or {}).get('snn_mean_synops_per_sample'))),
        'ann_flops_mean': _stat_mean(((comp.get('mean_synops_vs_mean_flops_per_sample') or {}).get('ann_flops_per_sample'))),
        't25_mean': _stat_mean(temporal.get('t25')),
        't50_mean': _stat_mean(temporal.get('t50')),
        't75_mean': _stat_mean(temporal.get('t75')),
        't100_mean': _stat_mean(temporal.get('t100')),
        't25_std': _stat_std(temporal.get('t25')),
        't50_std': _stat_std(temporal.get('t50')),
        't75_std': _stat_std(temporal.get('t75')),
        't100_std': _stat_std(temporal.get('t100')),
    })

results_df = pd.DataFrame(rows)
if results_df.empty:
    raise ValueError("No experiment rows found in research_sweep_results.json")

results_df = results_df.sort_values(["config_name", "dataset_name"]).reset_index(drop=True)
display(results_df)
print('Datasets found:', sorted(results_df['dataset_name'].dropna().unique().tolist()))
print('Modalities found:', sorted(results_df['modality'].dropna().unique().tolist()))


# Cross-modal completeness gate (baseline + structural_plasticity across vision + audio)
required_configs = ["baseline_snn", "structural_plasticity"]
required_modalities = ["vision", "audio"]
missing_pairs = []
for cfg in required_configs:
    for modality in required_modalities:
        subset = results_df[(results_df["config_name"] == cfg) & (results_df["modality"] == modality)]
        if subset.empty:
            missing_pairs.append(f"{cfg}:{modality}")

cross_modal_complete = len(missing_pairs) == 0
print('Cross-modal completeness:', cross_modal_complete)
if missing_pairs:
    print('Missing config/modality entries:', missing_pairs)

if REQUIRE_FULL_CROSS_MODAL_RESULTS and not cross_modal_complete:
    raise RuntimeError(
        "Cross-modal validation requires both datasets/modalities for baseline and structural_plasticity. "
        f"Missing: {missing_pairs}. Rerun Cell 9 with DATASETS=['ncaltech101', 'shd']."
    )


## Publication Outputs (LaTeX-Style Figures and Tables)

These cells export publication-ready artifacts from the notebook-generated `results_df` table and `experiment_map`. Run them after **Cell 12** (normalized results).


In [ ]:
"""Cell 12A: Publication Output Directories and Plot Style"""
from __future__ import annotations

from contextlib import contextmanager
from pathlib import Path

PUB_FIG_DIR = FIGURES_DIR / "publication"
PUB_TBL_DIR = RESULTS_DIR / "latex_tables"
PUB_FIG_DIR.mkdir(parents=True, exist_ok=True)
PUB_TBL_DIR.mkdir(parents=True, exist_ok=True)

LATEX_STYLE = {
    'font.family': 'serif',
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'legend.frameon': False,
    'grid.alpha': 0.2,
    # Keep TeX disabled for portability; use mathtext (LaTeX-like labels) instead.
    'text.usetex': False,
}

@contextmanager
def publication_style():
    with plt.rc_context(LATEX_STYLE):
        yield

print('Publication figure dir:', PUB_FIG_DIR)
print('LaTeX table dir:', PUB_TBL_DIR)


In [ ]:
"""Cell 12B: Export LaTeX Tables and Notebook-Generated Result Files"""
from __future__ import annotations

if "results_df" not in globals():
    raise RuntimeError("Run Cell 12 first to build results_df.")

# Compact summary table (publication-ready)
summary_cols = [
    "config_name", "dataset_name", "modality", "num_runs",
    "accuracy_mean", "accuracy_std",
    "energy_ratio_mean", "energy_ratio_std",
    "t25_mean", "t50_mean", "t75_mean", "t100_mean",
]
summary_table = results_df[summary_cols].copy().sort_values(["dataset_name", "config_name"])
summary_table = summary_table.rename(columns={
    'config_name': 'Config',
    'dataset_name': 'Dataset',
    'modality': 'Modality',
    'num_runs': 'Seeds',
    'accuracy_mean': r'Acc Mean (\%)',
    'accuracy_std': 'Acc Std',
    'energy_ratio_mean': 'Energy Ratio Mean',
    'energy_ratio_std': 'Energy Ratio Std',
    't25_mean': r'Sil@25\%',
    't50_mean': r'Sil@50\%',
    't75_mean': r'Sil@75\%',
    't100_mean': r'Sil@100\%',
})

# Round for clean exports
for col in summary_table.columns:
    if summary_table[col].dtype.kind in "fc":
        summary_table[col] = summary_table[col].round(4)

summary_csv = PUB_TBL_DIR / "cross_modal_results_summary.csv"
summary_tex = PUB_TBL_DIR / "cross_modal_results_summary.tex"
summary_table.to_csv(summary_csv, index=False)
summary_table.to_latex(summary_tex, index=False, float_format=lambda x: f"{x:.4f}")

# Peak temporal snapshot table
snapshot_cols = ["t25_mean", "t50_mean", "t75_mean", "t100_mean"]
peak_rows = []
for _, row in results_df.iterrows():
    vals = {c.replace("_mean", ""): float(row[c]) for c in snapshot_cols}
    peak_key = max(vals, key=vals.get)
    peak_rows.append({
        'Config': row['config_name'],
        'Dataset': row['dataset_name'],
        'Modality': row['modality'],
        'Peak Snapshot': peak_key,
        'Peak Silhouette': round(vals[peak_key], 4),
    })
peak_table = pd.DataFrame(peak_rows).sort_values(["Dataset", "Config"])
peak_csv = PUB_TBL_DIR / "cross_modal_temporal_peaks.csv"
peak_tex = PUB_TBL_DIR / "cross_modal_temporal_peaks.tex"
peak_table.to_csv(peak_csv, index=False)
peak_table.to_latex(peak_tex, index=False)

# Notebook-generated normalized results (machine-readable)
results_json = RESULTS_DIR / "notebook_generated_results_summary.json"
results_df.to_json(results_json, orient="records", indent=2)

print('Saved:', summary_csv)
print('Saved:', summary_tex)
print('Saved:', peak_csv)
print('Saved:', peak_tex)
print('Saved:', results_json)
display(summary_table)
display(peak_table)


In [ ]:
"""Cell 12C: Generate LaTeX-Style Cross-Modal Figures (PNG/PDF)"""
from __future__ import annotations

if "results_df" not in globals():
    raise RuntimeError("Run Cell 12 first to build results_df.")

SNAP_ORDER = ["t25", "t50", "t75", "t100"]
SNAP_LABELS = ["25%", "50%", "75%", "100%"]
CFG_COLORS = {"baseline_snn": "#1f77b4", "structural_plasticity": "#d62728"}
MOD_COLORS = {"vision": "#1f77b4", "audio": "#2ca02c"}
MOD_MARKERS = {"vision": "o", "audio": "s"}

# Figure 1: Temporal silhouette (one panel per config, vision vs audio)
cfgs = [c for c in ["baseline_snn", "structural_plasticity"] if c in set(results_df["config_name"].dropna())]
if not cfgs:
    raise ValueError("No baseline/structural_plasticity rows found in results_df")

with publication_style():
    fig, axes = plt.subplots(1, len(cfgs), figsize=(6.8 * len(cfgs), 4.8), squeeze=False)
    axes = axes.ravel()
    for ax, cfg in zip(axes, cfgs):
        sub = results_df[results_df["config_name"] == cfg].copy()
        for modality in ["vision", "audio"]:
            row = sub[sub["modality"] == modality]
            if row.empty:
                continue
            row = row.iloc[0]
            means = [float(row[f"{s}_mean"]) for s in SNAP_ORDER]
            stds = [float(row.get(f"{s}_std", 0.0)) for s in SNAP_ORDER]
            marker = MOD_MARKERS.get(modality, "o")
            color = MOD_COLORS.get(modality, CFG_COLORS.get(cfg))
            label = f"{modality.title()} ({row['dataset_name']})"
            x = np.arange(len(SNAP_ORDER))
            ax.errorbar(x, means, yerr=stds, marker=marker, linewidth=2.0, capsize=4, label=label, color=color)
        ax.axhline(0, linestyle="--", linewidth=1, color="black", alpha=0.5)
        ax.set_xticks(np.arange(len(SNAP_ORDER)))
        ax.set_xticklabels(SNAP_LABELS)
        ax.set_xlabel(r"Snapshot ($t/T_{max}$)")
        ax.set_ylabel(r"Silhouette Score $S_t$")
        ax.set_title(cfg.replace("_", " ").title())
        ax.legend(loc="best")
    fig.suptitle("Cross-Modal Temporal Engram Evolution", y=1.02, fontsize=14)
    fig.tight_layout()
    for ext in ["png", "pdf"]:
        out = PUB_FIG_DIR / f"cross_modal_temporal_engram_evolution.{ext}"
        fig.savefig(out, bbox_inches="tight")
        print("Saved:", out)
    plt.show()

# Figure 2: Accuracy + Energy ratio bar charts
with publication_style():
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.0))
    ax1, ax2 = axes
    sns.barplot(data=results_df, x="dataset_name", y="accuracy_mean", hue="config_name", ax=ax1)
    ax1.set_title("Accuracy by Dataset and Configuration")
    ax1.set_xlabel("Dataset")
    ax1.set_ylabel(r"Accuracy (\%)")
    for container in ax1.containers:
        try:
            ax1.bar_label(container, fmt="%.2f", padding=2, fontsize=8)
        except Exception:
            pass

    sns.barplot(data=results_df, x="dataset_name", y="energy_ratio_mean", hue="config_name", ax=ax2)
    ax2.axhline(TARGET_EFFICIENCY_RATIO, linestyle="--", linewidth=1.2, color="black", alpha=0.7)
    ax2.set_title("Energy Improvement Ratio (ANN / SNN)")
    ax2.set_xlabel("Dataset")
    ax2.set_ylabel("Ratio")
    for container in ax2.containers:
        try:
            ax2.bar_label(container, fmt="%.1f", padding=2, fontsize=8)
        except Exception:
            pass
    fig.tight_layout()
    for ext in ["png", "pdf"]:
        out = PUB_FIG_DIR / f"cross_modal_accuracy_energy.{ext}"
        fig.savefig(out, bbox_inches="tight")
        print("Saved:", out)
    plt.show()


In [ ]:
"""Cell 13: Cross-Modal Temporal Engram Comparison (Baseline + Structural Plasticity)"""
from __future__ import annotations

def find_experiment(config_name: str, modality: str):
    for key, exp in experiment_map.items():
        if not isinstance(exp, dict):
            continue
        if exp.get("config_name") == config_name and exp.get("modality") == modality:
            return key, exp
    return None, None

for cfg in ["baseline_snn", "structural_plasticity"]:
    v_key, vision_exp = find_experiment(cfg, "vision")
    a_key, audio_exp = find_experiment(cfg, "audio")
    print(f"{cfg}: vision={v_key}, audio={a_key}")
    if vision_exp is None or audio_exp is None:
        print(f"Skipping {cfg}: need both vision and audio results.")
        continue
    save_path = FIGURES_DIR / f"cross_modal_temporal__{cfg}.png"
    plot_engram_evolution(
        vision_results=vision_exp,
        audio_results=audio_exp,
        save_path=save_path,
        title=f"Temporal Engram Evolution ({cfg})",
    )
    print("Saved:", save_path)


In [ ]:
"""Cell 14: Cross-Modal Temporal Peak Snapshot Check"""
from __future__ import annotations

snapshot_cols = ["t25_mean", "t50_mean", "t75_mean", "t100_mean"]
peak_rows = []
for _, row in results_df.iterrows():
    vals = {c.replace("_mean", ""): row[c] for c in snapshot_cols}
    peak_key = max(vals, key=vals.get)
    peak_rows.append({
        'config_name': row['config_name'],
        'dataset_name': row['dataset_name'],
        'modality': row['modality'],
        'peak_snapshot': peak_key,
        'peak_silhouette': vals[peak_key],
    })
peak_df = pd.DataFrame(peak_rows).sort_values(["config_name", "dataset_name"])
display(peak_df)


In [ ]:
"""Cell 15: Accuracy and Energy Ratio Comparison Across Datasets"""
from __future__ import annotations

plot_df = results_df.copy()
if plot_df.empty:
    raise ValueError("results_df is empty")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
ax_acc, ax_energy = axes

sns.barplot(data=plot_df, x="dataset_name", y="accuracy_mean", hue="config_name", ax=ax_acc)
ax_acc.set_title("Accuracy by Dataset / Configuration")
ax_acc.set_xlabel("Dataset")
ax_acc.set_ylabel("Accuracy (%)")
for container in ax_acc.containers:
    try:
        ax_acc.bar_label(container, fmt="%.2f", padding=2, fontsize=9)
    except Exception:
        pass

sns.barplot(data=plot_df, x="dataset_name", y="energy_ratio_mean", hue="config_name", ax=ax_energy)
ax_energy.set_title("Energy Improvement Ratio (ANN / SNN)")
ax_energy.set_xlabel("Dataset")
ax_energy.set_ylabel("Ratio")
ax_energy.axhline(TARGET_EFFICIENCY_RATIO, linestyle="--", linewidth=1.5, color="black", alpha=0.7)
ax_energy.text(0.02, TARGET_EFFICIENCY_RATIO + 5, f"Target {TARGET_EFFICIENCY_RATIO:.0f}x", color="black")
for container in ax_energy.containers:
    try:
        ax_energy.bar_label(container, fmt="%.1f", padding=2, fontsize=9)
    except Exception:
        pass

plt.tight_layout()
out_path = FIGURES_DIR / "cross_modal_accuracy_energy_comparison.png"
plt.savefig(out_path, bbox_inches="tight")
plt.show()
print("Saved:", out_path)


In [ ]:
"""Cell 16: Export Compact Validation Summary (Optional)"""
from __future__ import annotations

summary_out = RESULTS_DIR / "cross_modal_validation_summary.csv"
cols = [
    "config_name", "dataset_name", "modality", "num_runs",
    "accuracy_mean", "accuracy_std",
    "energy_ratio_mean", "energy_ratio_std",
    "t25_mean", "t50_mean", "t75_mean", "t100_mean",
]
results_df[cols].to_csv(summary_out, index=False)
print("Saved:", summary_out)
